# Dome Flow Estimation from Louvers, Shutter, and Anemometers

This notebook builds a bulk-flow estimate for the Vera C. Rubin Observatory dome on Cerro Pachón.

## Ventilation System Overview

**Aperture shutter** — bi-parting door at the telescope pointing direction (0° reference):
- Without Light Wind Screen (LWS): ~200 m² effective open area
- With LWS installed: ~119 m² (110 m² optical gap + 90 m² panel area × 10% permeability)
- Contribution follows cos(θ) where θ = wind angle relative to shutter; contributes only within ±90° of the pointing direction

**Louvered Light Baffling Vents (LLBVs)** — 34 motorized louver assemblies in 12 bays spanning the full dome wall:
- 28 operational nighttime louvers from 53.1° to 306.9° (clockwise from shutter), ~232 m² total effective area
- 6 F/G bays at 180° (Rear Access Door, RAD) excluded from nighttime operations — always closed, not yet commissioned
- Sinusoidal baffle profile: ~80% effective airflow area at full-open; `effective_area_m2 = gross_area_m2 × 0.80`
- Individual effective areas range from 4.03 m² (Type 7, small 2-row) to 10.40 m² (Type 1, full 3-row)
- DCS controls each louver independently; upwind louvers act as inlets, downwind as exhaust to create cross-flow
- CFD target: ~2.5 m/s uniform interior flow

**Current commissioning configuration** opens the bottom row (row 1) of all non-RAD bays plus E2 and H2:
- Louvers: A1, B1, C1, D1, E1, E2, H1, H2, I1, L1, M1, N1 (~110 m² effective)

**Data sources:**
- Outside wind: `lsst.sal.ESS.airFlow` index 301
- Mount anemometers: `lsst.sal.ESS.airTurbulence` indices 110 and 123–126
- Dome azimuth: `lsst.sal.MTDome.azimuth`
- Shutter opening: `lsst.sal.MTDome.apertureShutter`
- Louver geometry: `louver_map.csv` (34 louvers with azimuths, effective areas, RAD flags)

## Model Summary

For each timestamp, the notebook computes:

- Windward louver area: the sum of louver areas facing the incoming wind
- Leeward louver area: the sum of louver areas facing away from the incoming wind
- Shutter area contribution from the measured shutter opening fraction
- Effective through area: `min(A_in, A_out)` as a bottleneck approximation
- Outside driver: `U_out * A_eff`
- Inside flow proxy: mean or median mount-anemometer speed

If you also provide an internal reference area, the notebook converts the inside proxy to an estimated volumetric flow and fits an effective discharge coefficient.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from pathlib import Path
from IPython.display import display

from astropy.time import Time, TimeDelta
import astropy.units as u

from lsst_efd_client import EfdClient

%matplotlib inline

In [ ]:
EFD_ALIAS = "usdf_efd"

t_end = Time("2026-04-01T05:00:00", scale="utc")
t_start = t_end - TimeDelta(25 * u.day)

RESAMPLE_RULE = "30s"
MIN_INSIDE_SENSORS = 3

flow_topic = "lsst.sal.ESS.airFlow"
turb_topic = "lsst.sal.ESS.airTurbulence"
dome_topic = "lsst.sal.MTDome.azimuth"
shutter_topic = "lsst.sal.MTDome.apertureShutter"

weather_index = 301
speed_field = "speedMagnitude"
outside_speed_field = "speed"
outside_direction_field = "direction"
dome_azimuth_field = "positionActual"
shutter_fields = ["positionActual0", "positionActual1"]

ess_sensors = {
    110: "TMA Platform",
    123: "Top Ring -X/-Y",
    124: "Top Ring +X/-Y",
    125: "Top Ring +X/+Y",
    126: "Top Ring -X/+Y",
}

speed_clip = {
    110: None,
    123: 8.6,
    124: 8.6,
    125: 8.6,
    126: 8.6,
}

# ── Shutter geometry ──────────────────────────────────────────────────────────
# Without LWS: ~200 m² full open.
# With LWS installed: ~119 m² (110 m² optical gap + 90 m² panel area × 10% air permeability).
SHUTTER_FULL_OPEN_AREA_M2_NO_LWS = 200.0
SHUTTER_FULL_OPEN_AREA_M2_WITH_LWS = 119.0
SHUTTER_FULL_OPEN_AREA_M2 = (
    SHUTTER_FULL_OPEN_AREA_M2_NO_LWS  # switch to _WITH_LWS when LWS is installed
)

SHUTTER_AZIMUTH_OFFSET_DEG = 0.0

# ── Internal flow calibration (set when flow measurements are available) ──────
INTERNAL_REFERENCE_AREA_M2 = np.nan
INITIAL_DISCHARGE_COEFFICIENT = 1.0

print(f"Start: {t_start.iso}")
print(f"End:   {t_end.iso}")
print(f"Resample rule: {RESAMPLE_RULE}")
print(
    f"Shutter area: {SHUTTER_FULL_OPEN_AREA_M2:.0f} m²  (no-LWS={SHUTTER_FULL_OPEN_AREA_M2_NO_LWS:.0f}, with-LWS={SHUTTER_FULL_OPEN_AREA_M2_WITH_LWS:.0f})"
)

## Louver Geometry Input

The notebook looks for a file named `louver_map.csv` in the working directory. If it does not exist, it creates a 34-row template.

Required columns:

- `louver_id`: label such as `L01`
- `azimuth_dome_deg`: louver outward normal in dome coordinates, degrees
- `full_open_area_m2`: full-open geometric area in square meters
- `open_fraction`: static opening fraction between 0 and 1 for the analysis window

If only some louvers are open, set the others to `0.0`.

In [ ]:
louver_map_path = Path("../data/louver_map.csv")

if louver_map_path.exists():
    louver_map = pd.read_csv(louver_map_path)
else:
    louver_map = pd.DataFrame(
        {
            "louver_id": [f"L{i:02d}" for i in range(1, 35)],
            "louver_type": [np.nan] * 34,
            "gross_area_m2": [np.nan] * 34,
            "effective_area_m2": [np.nan]
            * 34,  # gross × 0.80 (sinusoidal baffle factor)
            "azimuth_dome_deg": [np.nan] * 34,
            "is_rad": [False]
            * 34,  # True for F/G RAD bays (180°, excluded from nighttime ops)
            "open_fraction": [1.0] * 34,
        }
    )

# Ensure boolean type for is_rad (CSV may read it as str)
louver_map["is_rad"] = louver_map["is_rad"].astype(bool)

nighttime_louvers = louver_map[~louver_map["is_rad"]]
commissioning_open = louver_map[louver_map["open_fraction"] > 0]

display(louver_map)
print()
print(
    f"Total louvers: {len(louver_map)}  |  Nighttime operational (non-RAD): {(~louver_map['is_rad']).sum()}"
)
print(f"Open in current config: {(louver_map['open_fraction'] > 0).sum()} louvers")
print(f"  ids: {', '.join(commissioning_open['louver_id'].tolist())}")
print(
    f"  effective louver area (open only): {(louver_map['effective_area_m2'] * louver_map['open_fraction']).sum():.1f} m²"
)
print(
    f"  gross louver area (open only):     {(louver_map['gross_area_m2'] * louver_map['open_fraction']).sum():.1f} m²"
)
print(
    f"Total nighttime effective area (all non-RAD fully open): {nighttime_louvers['effective_area_m2'].sum():.1f} m²"
)

In [ ]:
def wrap180(angle_deg):
    angle_deg = np.asarray(angle_deg, dtype=float)
    return (angle_deg + 180.0) % 360.0 - 180.0


def circular_mean_deg(values):
    values = pd.Series(values).dropna().astype(float)
    if values.empty:
        return np.nan
    radians = np.deg2rad(values.to_numpy())
    return np.mod(
        np.rad2deg(np.arctan2(np.sin(radians).mean(), np.cos(radians).mean())), 360.0
    )


def validate_geometry(louver_map, shutter_full_open_area_m2):
    required_cols = [
        "louver_id",
        "effective_area_m2",
        "azimuth_dome_deg",
        "open_fraction",
        "is_rad",
    ]
    missing_cols = [col for col in required_cols if col not in louver_map.columns]
    if missing_cols:
        raise ValueError(f"louver_map is missing columns: {missing_cols}")

    if len(louver_map) != 34:
        raise ValueError(f"Expected 34 louvers, found {len(louver_map)}")

    if (
        louver_map[["effective_area_m2", "azimuth_dome_deg", "open_fraction"]]
        .isna()
        .any()
        .any()
    ):
        raise ValueError(
            "Fill effective_area_m2, azimuth_dome_deg, and open_fraction for all 34 louvers."
        )

    if (louver_map["open_fraction"] < 0).any() or (
        louver_map["open_fraction"] > 1
    ).any():
        raise ValueError("open_fraction must be between 0 and 1")

    if not np.isfinite(shutter_full_open_area_m2) or shutter_full_open_area_m2 <= 0:
        raise ValueError(
            "Set SHUTTER_FULL_OPEN_AREA_M2 to the full-open shutter aperture area in m^2"
        )

    n_rad = louver_map["is_rad"].sum()
    if n_rad != 6:
        print(f"Warning: expected 6 RAD louvers (F/G at 180°), found {n_rad}")


def resample_direction(series, rule):
    return series.resample(rule).apply(circular_mean_deg)


def prepare_inside_sensor(df, sensor_idx, label, clip_threshold, rule):
    speed = df[speed_field].astype(float).copy()
    if clip_threshold is not None:
        speed = speed.where(speed < clip_threshold)
    speed = speed.resample(rule).mean()
    speed.name = f"{sensor_idx}_{label}"
    return speed


def compute_effective_areas(
    frame, louver_map, shutter_full_open_area_m2, shutter_azimuth_offset_deg
):
    """Compute per-timestamp windward, leeward, shutter, and effective through areas.

    Uses effective_area_m2 from louver_map (= gross × 0.80 sinusoidal baffle factor).
    RAD louvers (is_rad=True) are excluded by their open_fraction=0 in the CSV.
    """
    louver_angles = louver_map["azimuth_dome_deg"].to_numpy(dtype=float)
    louver_areas = louver_map["effective_area_m2"].to_numpy(dtype=float) * louver_map[
        "open_fraction"
    ].to_numpy(dtype=float)

    windward_area = np.full(len(frame), np.nan)
    leeward_area = np.full(len(frame), np.nan)
    shutter_area = np.full(len(frame), np.nan)
    effective_through_area = np.full(len(frame), np.nan)

    wind_dir = frame["outside_wind_dir_deg"].to_numpy(dtype=float)
    dome_az = frame["dome_azimuth_deg"].to_numpy(dtype=float)
    shutter_frac = frame["shutter_open_fraction"].to_numpy(dtype=float)

    for i in range(len(frame)):
        if (
            not np.isfinite(wind_dir[i])
            or not np.isfinite(dome_az[i])
            or not np.isfinite(shutter_frac[i])
        ):
            continue

        global_louver_angles = dome_az[i] + louver_angles
        rel = wrap180(wind_dir[i] - global_louver_angles)
        projection = np.cos(np.deg2rad(rel))

        a_in_louver = np.sum(louver_areas * np.clip(projection, 0, None))
        a_out_louver = np.sum(louver_areas * np.clip(-projection, 0, None))

        shutter_angle = dome_az[i] + shutter_azimuth_offset_deg
        shutter_rel = wrap180(wind_dir[i] - shutter_angle)
        shutter_proj = np.cos(np.deg2rad(shutter_rel))
        a_shutter = shutter_full_open_area_m2 * shutter_frac[i]

        a_in_shutter = a_shutter * max(shutter_proj, 0.0)
        a_out_shutter = a_shutter * max(-shutter_proj, 0.0)

        windward_area[i] = a_in_louver + a_in_shutter
        leeward_area[i] = a_out_louver + a_out_shutter
        shutter_area[i] = a_shutter
        effective_through_area[i] = min(windward_area[i], leeward_area[i])

    result = frame.copy()
    result["windward_open_area_m2"] = windward_area
    result["leeward_open_area_m2"] = leeward_area
    result["shutter_open_area_m2"] = shutter_area
    result["effective_through_area_m2"] = effective_through_area
    result["outside_driver_m3s_cd1"] = (
        result["outside_wind_speed_mps"] * result["effective_through_area_m2"]
    )
    return result

In [ ]:
validate_geometry(louver_map, SHUTTER_FULL_OPEN_AREA_M2)
print("Geometry validation passed.")
open_eff = np.sum(louver_map["effective_area_m2"] * louver_map["open_fraction"])
print(f"Effective louver area at current open fractions: {open_eff:.1f} m²")
print(f"Shutter area (active config): {SHUTTER_FULL_OPEN_AREA_M2:.0f} m²")

In [ ]:
client = EfdClient(EFD_ALIAS)
print(f"Connected to {EFD_ALIAS}")

df_weather = await client.select_time_series(
    flow_topic,
    fields=[outside_speed_field, outside_direction_field],
    start=t_start,
    end=t_end,
    index=weather_index,
)

df_dome = await client.select_time_series(
    dome_topic,
    fields=[dome_azimuth_field],
    start=t_start,
    end=t_end,
)

df_shutter = await client.select_time_series(
    shutter_topic,
    fields=shutter_fields,
    start=t_start,
    end=t_end,
)

inside_raw = {}
for sensor_idx, label in ess_sensors.items():
    inside_raw[sensor_idx] = await client.select_time_series(
        turb_topic,
        fields=[speed_field],
        start=t_start,
        end=t_end,
        index=sensor_idx,
    )

print(f"Weather rows: {len(df_weather)}")
print(f"Dome azimuth rows: {len(df_dome)}")
print(f"Shutter rows: {len(df_shutter)}")
for sensor_idx, label in ess_sensors.items():
    print(
        f"Inside sensor {sensor_idx:3d} {label:16s}: {len(inside_raw[sensor_idx])} rows"
    )

In [ ]:
weather_aligned = pd.DataFrame(
    {
        "outside_wind_speed_mps": df_weather[outside_speed_field]
        .astype(float)
        .resample(RESAMPLE_RULE)
        .mean(),
        "outside_wind_dir_deg": resample_direction(
            df_weather[outside_direction_field], RESAMPLE_RULE
        ),
    }
)

dome_aligned = pd.DataFrame(
    {
        "dome_azimuth_deg": resample_direction(
            df_dome[dome_azimuth_field], RESAMPLE_RULE
        ),
    }
)

shutter_aligned = (
    df_shutter[shutter_fields]
    .astype(float)
    .resample(RESAMPLE_RULE)
    .mean()
    .rename(
        columns={
            "positionActual0": "shutter_left_pct",
            "positionActual1": "shutter_right_pct",
        }
    )
)
shutter_aligned["shutter_open_fraction"] = (
    0.5
    * (shutter_aligned["shutter_left_pct"] + shutter_aligned["shutter_right_pct"])
    / 100.0
)

inside_series = []
for sensor_idx, label in ess_sensors.items():
    sensor_series = prepare_inside_sensor(
        inside_raw[sensor_idx],
        sensor_idx=sensor_idx,
        label=label.replace(" ", "_"),
        clip_threshold=speed_clip[sensor_idx],
        rule=RESAMPLE_RULE,
    )
    inside_series.append(sensor_series)

inside_aligned = pd.concat(inside_series, axis=1)
inside_aligned["inside_speed_mean_mps"] = inside_aligned.mean(axis=1, skipna=True)
inside_aligned["inside_speed_median_mps"] = inside_aligned.median(axis=1, skipna=True)
inside_aligned["inside_sensor_count"] = inside_aligned.notna().sum(axis=1)

model_df = pd.concat(
    [weather_aligned, dome_aligned, shutter_aligned, inside_aligned], axis=1
)
model_df = model_df.sort_index()
model_df = model_df[model_df["inside_sensor_count"] >= MIN_INSIDE_SENSORS].copy()

model_df = compute_effective_areas(
    model_df,
    louver_map=louver_map,
    shutter_full_open_area_m2=SHUTTER_FULL_OPEN_AREA_M2,
    shutter_azimuth_offset_deg=SHUTTER_AZIMUTH_OFFSET_DEG,
)

model_df["inside_flow_proxy_m3s"] = (
    model_df["inside_speed_mean_mps"] * INTERNAL_REFERENCE_AREA_M2
)
model_df["outside_flow_initial_m3s"] = (
    INITIAL_DISCHARGE_COEFFICIENT * model_df["outside_driver_m3s_cd1"]
)

print(f"Aligned samples after sensor-count filter: {len(model_df)}")
display(model_df.head())

In [ ]:
fit_df = model_df.dropna(
    subset=[
        "outside_wind_speed_mps",
        "effective_through_area_m2",
        "inside_speed_mean_mps",
    ]
).copy()
fit_df = fit_df[fit_df["effective_through_area_m2"] > 0].copy()

x = fit_df["outside_driver_m3s_cd1"].to_numpy(dtype=float)
y_speed = fit_df["inside_speed_mean_mps"].to_numpy(dtype=float)

speed_gain = np.dot(x, y_speed) / np.dot(x, x)
fit_df["inside_speed_fit_mps"] = speed_gain * fit_df["outside_driver_m3s_cd1"]
speed_corr = np.corrcoef(x, y_speed)[0, 1] if len(fit_df) > 1 else np.nan

print(f"Samples used for fit: {len(fit_df)}")
print(f"Best-fit speed gain [m/s per (m^3/s at Cd=1)]: {speed_gain:.6f}")
print(f"Correlation between inside speed and outside driver: {speed_corr:.3f}")

if np.isfinite(INTERNAL_REFERENCE_AREA_M2) and INTERNAL_REFERENCE_AREA_M2 > 0:
    y_flow = fit_df["inside_flow_proxy_m3s"].to_numpy(dtype=float)
    fitted_cd = np.dot(x, y_flow) / np.dot(x, x)
    fit_df["outside_flow_fit_m3s"] = fitted_cd * fit_df["outside_driver_m3s_cd1"]
    flow_corr = np.corrcoef(x, y_flow)[0, 1] if len(fit_df) > 1 else np.nan
    print(f"Fitted discharge coefficient: {fitted_cd:.4f}")
    print(f"Correlation in flow units: {flow_corr:.3f}")
else:
    fitted_cd = np.nan
    print(
        "INTERNAL_REFERENCE_AREA_M2 is not set, so the notebook fits only a speed gain and not an absolute flow coefficient."
    )

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(
    model_df.index, model_df["outside_wind_speed_mps"], color="tab:blue", lw=0.9
)
axes[0].set_ylabel("Outside wind\n(m/s)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    model_df.index,
    model_df["inside_speed_mean_mps"],
    color="tab:orange",
    lw=0.9,
    label="Inside mean",
)
axes[1].plot(
    model_df.index,
    model_df["inside_speed_median_mps"],
    color="tab:red",
    lw=0.9,
    alpha=0.7,
    label="Inside median",
)
axes[1].set_ylabel("Inside speed\n(m/s)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    model_df.index,
    model_df["windward_open_area_m2"],
    color="tab:green",
    lw=0.9,
    label="Windward",
)
axes[2].plot(
    model_df.index,
    model_df["leeward_open_area_m2"],
    color="tab:purple",
    lw=0.9,
    label="Leeward",
)
axes[2].plot(
    model_df.index,
    model_df["effective_through_area_m2"],
    color="black",
    lw=1.1,
    label="Effective through area",
)
axes[2].set_ylabel("Area\n(m^2)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(
    model_df.index,
    model_df["outside_driver_m3s_cd1"],
    color="tab:brown",
    lw=0.9,
    label="U_out * A_eff",
)
if np.isfinite(INTERNAL_REFERENCE_AREA_M2) and INTERNAL_REFERENCE_AREA_M2 > 0:
    axes[3].plot(
        model_df.index,
        model_df["inside_flow_proxy_m3s"],
        color="tab:cyan",
        lw=0.9,
        alpha=0.8,
        label="Inside flow proxy",
    )
    axes[3].set_ylabel("Flow proxy\n(m^3/s)")
else:
    axes[3].plot(
        model_df.index,
        model_df["inside_speed_mean_mps"],
        color="tab:cyan",
        lw=0.9,
        alpha=0.8,
        label="Inside speed mean",
    )
    axes[3].set_ylabel("Driver / speed\n(mixed units)")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    fit_df["outside_driver_m3s_cd1"],
    fit_df["inside_speed_mean_mps"],
    s=8,
    alpha=0.25,
    c=fit_df["shutter_open_fraction"],
    cmap="viridis",
)

x_line = np.linspace(0, fit_df["outside_driver_m3s_cd1"].quantile(0.99), 200)
ax.plot(
    x_line, speed_gain * x_line, color="black", lw=2.0, label="Best-fit speed relation"
)

ax.set_xlabel("Outside driver: U_out * A_eff [m^3/s at Cd=1]")
ax.set_ylabel("Inside mount-anemometer mean speed (m/s)")
ax.set_title("Inside speed vs outside wind-area driver")
ax.grid(True, alpha=0.3)
ax.legend()
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label("Shutter open fraction")
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.Series(
    {
        "samples": len(fit_df),
        "outside_wind_mean_mps": fit_df["outside_wind_speed_mps"].mean(),
        "inside_speed_mean_mps": fit_df["inside_speed_mean_mps"].mean(),
        "effective_area_mean_m2": fit_df["effective_through_area_m2"].mean(),
        "effective_area_p90_m2": fit_df["effective_through_area_m2"].quantile(0.90),
        "outside_driver_mean_m3s_cd1": fit_df["outside_driver_m3s_cd1"].mean(),
        "speed_gain": speed_gain,
        "fitted_discharge_coefficient": fitted_cd,
    }
)
display(summary.to_frame(name="value"))

display(
    fit_df[
        [
            "outside_wind_speed_mps",
            "outside_wind_dir_deg",
            "dome_azimuth_deg",
            "shutter_open_fraction",
            "windward_open_area_m2",
            "leeward_open_area_m2",
            "effective_through_area_m2",
            "inside_speed_mean_mps",
            "outside_driver_m3s_cd1",
        ]
    ].head(20)
)

In [ ]:
lmap = louver_map.copy()
if "louver_type" not in lmap.columns:
    raise ValueError("louver_type column not found in louver_map.csv.")

lmap["louver_type"] = lmap["louver_type"].astype(int)
lmap["active_area_m2"] = lmap["effective_area_m2"] * lmap["open_fraction"]

theta = np.deg2rad(
    model_df["outside_wind_dir_deg"].to_numpy(dtype=float)[:, None]
    - (
        model_df["dome_azimuth_deg"].to_numpy(dtype=float)[:, None]
        + lmap["azimuth_dome_deg"].to_numpy(dtype=float)[None, :]
    )
)
proj = np.cos(theta)
ain_mat = np.clip(proj, 0, None) * lmap["active_area_m2"].to_numpy(dtype=float)[None, :]
aout_mat = (
    np.clip(-proj, 0, None) * lmap["active_area_m2"].to_numpy(dtype=float)[None, :]
)

type_rows = []
for t in sorted(lmap["louver_type"].unique()):
    mask = lmap["louver_type"].to_numpy() == t
    is_rad_type = lmap.loc[mask, "is_rad"].any()
    type_rows.append(
        {
            "louver_type": int(t),
            "count": int(mask.sum()),
            "is_rad": bool(is_rad_type),
            "static_eff_area_m2": float(lmap.loc[mask, "active_area_m2"].sum()),
            "mean_windward_contrib_m2": float(np.nanmean(ain_mat[:, mask].sum(axis=1))),
            "mean_leeward_contrib_m2": float(np.nanmean(aout_mat[:, mask].sum(axis=1))),
        }
    )

type_diag = pd.DataFrame(type_rows).sort_values("louver_type").reset_index(drop=True)
display(type_diag)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(type_diag))
w = 0.26
ax.bar(x - w, type_diag["static_eff_area_m2"], width=w, label="Static effective area")
ax.bar(
    x,
    type_diag["mean_windward_contrib_m2"],
    width=w,
    label="Mean windward contribution",
)
ax.bar(
    x + w,
    type_diag["mean_leeward_contrib_m2"],
    width=w,
    label="Mean leeward contribution",
)
for xi, row in zip(x, type_diag.itertuples()):
    if row.is_rad:
        ax.axvspan(
            xi - 2 * w,
            xi + 2 * w,
            alpha=0.12,
            color="red",
            label="_RAD" if xi == x[0] else "",
        )
ax.set_xticks(x)
ax.set_xticklabels(
    [
        f"T{int(t)}" + (" (RAD)" if r else "")
        for t, r in zip(type_diag["louver_type"], type_diag["is_rad"])
    ]
)
ax.set_xlabel("Louver type")
ax.set_ylabel("Effective area (m²)")
ax.set_title("Louver type diagnostics (effective areas, RAD types shaded)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Airflow Efficiency vs Relative Wind Direction

Defines airflow efficiency as the inside-to-outside speed ratio:

`airflow_efficiency = inside_speed_mean_mps / outside_wind_speed_mps`

and plots it against relative wind direction:

`relative_wind_deg = wrap180(outside_wind_dir_deg - (dome_azimuth_deg + SHUTTER_AZIMUTH_OFFSET_DEG))`

In [ ]:
eff_df = model_df.copy()

eff_df["relative_wind_deg"] = wrap180(
    eff_df["outside_wind_dir_deg"]
    - (eff_df["dome_azimuth_deg"] + SHUTTER_AZIMUTH_OFFSET_DEG)
)

# Use median of top-ring sensors only (123–126), excluding TMA Platform (110)
topring_cols_6 = [
    c
    for c in eff_df.columns
    if any(c.startswith(f"{idx}_") for idx in [123, 124, 125, 126])
]
eff_df["inside_speed_topring_mps"] = eff_df[topring_cols_6].median(axis=1, skipna=True)
eff_df["airflow_efficiency"] = (
    eff_df["inside_speed_topring_mps"] / eff_df["outside_wind_speed_mps"]
)

eff_df = eff_df[
    np.isfinite(eff_df["relative_wind_deg"])
    & np.isfinite(eff_df["airflow_efficiency"])
    & np.isfinite(eff_df["outside_wind_speed_mps"])
    & np.isfinite(eff_df["shutter_open_fraction"])
    & (eff_df["shutter_open_fraction"] > 0.95)
    & (eff_df["outside_wind_speed_mps"] > 0.5)
    & (eff_df["airflow_efficiency"] >= 0)
    & (eff_df["airflow_efficiency"] <= 3.0)
].copy()

fig, ax1 = plt.subplots(figsize=(13, 6))
sc = ax1.scatter(
    eff_df["relative_wind_deg"],
    eff_df["airflow_efficiency"],
    c=eff_df["outside_wind_speed_mps"],
    cmap="viridis",
    s=8,
    alpha=0.35,
    zorder=2,
)

bins = np.arange(-180, 181, 15)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_idx = np.digitize(eff_df["relative_wind_deg"], bins) - 1
med = [
    eff_df.loc[bin_idx == i, "airflow_efficiency"].median() for i in range(len(centers))
]
ax1.plot(centers, med, color="black", lw=2.2, label="15° bin median", zorder=4)

ax1.set_xlabel("Relative wind direction (° from shutter, 0° = wind into shutter)")
ax1.set_ylabel(
    "Airflow efficiency (inside / outside speed)\n[top-ring sensors 123–126 median]"
)
ax1.set_xlim(-180, 180)
ax1.set_xticks(np.arange(-180, 181, 30))
ax1.set_ylim(0, None)
ax1.grid(True, alpha=0.3)

cbar = plt.colorbar(sc, ax=ax1)
cbar.set_label("Outside wind speed (m/s)")

# ── Right axis: 6-louver windward aperture model ──────────────────────────────
ax2 = ax1.twinx()
ax2.plot(
    wind_deg,
    shutter_no_lws,
    color="tab:cyan",
    lw=1.4,
    ls="--",
    label=f"Shutter only, no LWS ({S_NO_LWS:.0f} m²)",
    zorder=3,
)
ax2.plot(
    wind_deg,
    wt_6_no_lws,
    color="tab:purple",
    lw=2.2,
    label=f"Shutter (no LWS) + 6-louver config  [{louver_eff_6:.0f}+{S_NO_LWS:.0f} = {louver_eff_6+S_NO_LWS:.0f} m²]",
    zorder=3,
)
ax2.set_ylabel("Windward open area (m²)", color="tab:purple")
ax2.tick_params(axis="y", labelcolor="tab:purple")
ax2.set_ylim(0, None)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

ax1.set_title(
    "Airflow efficiency vs relative wind direction\n"
    "6-louver commissioning config (B1, E1, E2, H1, H2, M1)  |  inside = median(ESS 123–126)"
)
plt.tight_layout()
plt.show()

print(f"Top-ring columns used: {topring_cols_6}")
print(f"Points used: {len(eff_df)}")
print(f"Median efficiency: {eff_df['airflow_efficiency'].median():.3f}")
print(
    f"P10-P90 efficiency: {eff_df['airflow_efficiency'].quantile(0.10):.3f} – {eff_df['airflow_efficiency'].quantile(0.90):.3f}"
)

## Next Refinements

Useful follow-ups once the notebook is running with your geometry map:

- Replace the static `open_fraction` column with time-dependent louver states if those are available
- Split the model by wind-relative dome angle, not just raw outside speed
- Compare top-ring sensors individually instead of only the across-sensor mean
- Fit separate coefficients for windward-only, shutter-only, and mixed aperture cases
- Add exposure metadata and image-quality metrics if you want to link flow state to observing performance

## Effective Area vs Relative Wind Bearing

Plots the effective louver and shutter area as a function of wind bearing angle relative to the shutter.

In [ ]:
def compute_windward_total(
    wind_angles, louver_azimuths, louver_areas, shutter_full_open_area_m2
):
    """Compute windward total area (louver_in + shutter_in) for a sweep of wind angles."""
    n = len(wind_angles)
    out = np.zeros(n)
    for i, wind_angle in enumerate(wind_angles):
        rel = wrap180(wind_angle - louver_azimuths)
        lw = np.sum(louver_areas * np.clip(np.cos(np.deg2rad(rel)), 0, None))
        s_in = shutter_full_open_area_m2 * max(np.cos(np.deg2rad(wind_angle)), 0.0)
        out[i] = lw + s_in
    return out


def active_areas_for(louver_map, open_ids):
    """Return effective_area_m2 array weighted to 1.0 for louvers in open_ids."""
    fracs = np.where(louver_map["louver_id"].isin(open_ids), 1.0, 0.0)
    return louver_map["effective_area_m2"].to_numpy(dtype=float) * fracs


# Sweep 0–360° then sort into –180…180 for a centred x-axis
wind_angles_raw = np.linspace(0, 360, 721)
sort_idx = np.argsort(wrap180(wind_angles_raw))
wind_deg = wrap180(wind_angles_raw)[sort_idx]
wind_angles = wind_angles_raw[sort_idx]

laz = louver_map["azimuth_dome_deg"].to_numpy(dtype=float)

# ── Louver sets ───────────────────────────────────────────────────────────────
ids_6 = ["B1", "E1", "E2", "H1", "H2", "M1"]
ids_12 = ["A1", "B1", "C1", "D1", "E1", "E2", "H1", "H2", "I1", "L1", "M1", "N1"]
ids_34 = louver_map["louver_id"].tolist()  # all 34 including RAD

S_NO_LWS = SHUTTER_FULL_OPEN_AREA_M2_NO_LWS
S_WITH_LWS = SHUTTER_FULL_OPEN_AREA_M2_WITH_LWS

shutter_no_lws = S_NO_LWS * np.clip(np.cos(np.deg2rad(wind_angles)), 0, None)
shutter_with_lws = S_WITH_LWS * np.clip(np.cos(np.deg2rad(wind_angles)), 0, None)

wt_6_no_lws = compute_windward_total(
    wind_angles, laz, active_areas_for(louver_map, ids_6), S_NO_LWS
)
wt_12_no_lws = compute_windward_total(
    wind_angles, laz, active_areas_for(louver_map, ids_12), S_NO_LWS
)
wt_12_lws = compute_windward_total(
    wind_angles, laz, active_areas_for(louver_map, ids_12), S_WITH_LWS
)
wt_34_lws = compute_windward_total(
    wind_angles, laz, active_areas_for(louver_map, ids_34), S_WITH_LWS
)

louver_eff_6 = active_areas_for(louver_map, ids_6).sum()
louver_eff_12 = active_areas_for(louver_map, ids_12).sum()
louver_eff_34 = active_areas_for(louver_map, ids_34).sum()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 14), sharex=True, sharey=False)
xticks = np.arange(-180, 181, 30)


def _fmt(ax, title):
    ax.set_xlim(-180, 180)
    ax.set_xticks(xticks)
    ax.axvline(0, color="k", lw=0.8, ls=":", alpha=0.4)
    ax.set_ylabel("Windward open area (m²)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)


# ── Panel 1: shutter no-LWS + 6 louvers ──────────────────────────────────────
ax = axes[0]
ax.plot(
    wind_deg,
    shutter_no_lws,
    color="tab:cyan",
    lw=1.5,
    ls="--",
    label=f"Shutter only, no LWS ({S_NO_LWS:.0f} m²)",
)
ax.plot(
    wind_deg,
    wt_6_no_lws,
    color="tab:purple",
    lw=2.2,
    label=f"Shutter (no LWS) + 6-louver config  [{louver_eff_6:.0f}+{S_NO_LWS:.0f} = {louver_eff_6+S_NO_LWS:.0f} m²]",
)
_fmt(
    ax, f"Panel 1 — Shutter (no LWS, {S_NO_LWS:.0f} m²) + 6-louver commissioning config"
)

# ── Panel 2: shutter with/without LWS + 12 louvers ───────────────────────────
ax = axes[1]
ax.plot(
    wind_deg,
    shutter_no_lws,
    color="tab:cyan",
    lw=1.5,
    ls="--",
    label=f"Shutter only, no LWS ({S_NO_LWS:.0f} m²)",
)
ax.plot(
    wind_deg,
    shutter_with_lws,
    color="tab:cyan",
    lw=1.5,
    ls="-.",
    label=f"Shutter only, with LWS ({S_WITH_LWS:.0f} m²)",
)
ax.plot(
    wind_deg,
    wt_12_no_lws,
    color="tab:red",
    lw=2.2,
    label=f"Shutter (no LWS) + 12-louver config  [{louver_eff_12:.0f}+{S_NO_LWS:.0f} = {louver_eff_12+S_NO_LWS:.0f} m²]",
)
ax.plot(
    wind_deg,
    wt_12_lws,
    color="tab:orange",
    lw=2.2,
    label=f"Shutter (LWS) + 12-louver config  [{louver_eff_12:.0f}+{S_WITH_LWS:.0f} = {louver_eff_12+S_WITH_LWS:.0f} m²]",
)
_fmt(ax, f"Panel 2 — 12-louver config (A1–N1 excl. F/G + E2, H2) with and without LWS")

# ── Panel 3: shutter with LWS + all 34 louvers ───────────────────────────────
ax = axes[2]
ax.plot(
    wind_deg,
    shutter_with_lws,
    color="tab:cyan",
    lw=1.5,
    ls="-.",
    label=f"Shutter only, with LWS ({S_WITH_LWS:.0f} m²)",
)
ax.plot(
    wind_deg,
    wt_34_lws,
    color="tab:gray",
    lw=2.2,
    label=f"Shutter (LWS) + all 34 louvers  [{louver_eff_34:.0f}+{S_WITH_LWS:.0f} = {louver_eff_34+S_WITH_LWS:.0f} m²]",
)
_fmt(ax, f"Panel 3 — Shutter (with LWS, {S_WITH_LWS:.0f} m²) + all 34 louvers")

axes[-1].set_xlabel(
    "Wind bearing angle (° clockwise from shutter, 0° = shutter facing wind)"
)
fig.suptitle(
    "Dome windward aperture area vs relative wind direction", fontsize=12, y=1.005
)
plt.tight_layout()
plt.show()

In [ ]:
# This cell is superseded by the updated standalone 6-louver efficiency plot above.
# Kept for reference only — delete or skip as needed.
print(
    "Skipping duplicate overlay cell — see standalone 6-louver efficiency plot above."
)

## Airflow Efficiency: 12-Louver Nights (2026-04-23, -25, -26, -27)

Four nights with the expanded commissioning configuration:
**A1, B1, C1, D1, E1, E2, H1, H2, I1, L1, M1, N1** (~110 m² effective louver area).

Points are coloured by night; the 15° binned median is overlaid.
The right axis shows the 12-louver windward-aperture model for reference.

In [ ]:
# ── 12-louver config: override open_fraction from CSV (currently 6-louver) ────
ids_12lou = ["A1", "B1", "C1", "D1", "E1", "E2", "H1", "H2", "I1", "L1", "M1", "N1"]
louver_map_12 = louver_map.copy()
louver_map_12["open_fraction"] = np.where(
    louver_map_12["louver_id"].isin(ids_12lou), 1.0, 0.0
)
louver_eff_12lou = (
    louver_map_12["effective_area_m2"] * louver_map_12["open_fraction"]
).sum()
print(f"12-louver effective area: {louver_eff_12lou:.1f} m²")
print(
    f"  open ids: {', '.join(louver_map_12[louver_map_12['open_fraction'] > 0]['louver_id'])}"
)

# ── Night boundaries (UTC; Cerro Pachón UTC-4, nights start ~22:00 UTC) ───────
nights_12lou = {
    "2026-04-23": (
        Time("2026-04-23T22:00:00", scale="utc"),
        Time("2026-04-24T11:00:00", scale="utc"),
    ),
    "2026-04-25": (
        Time("2026-04-25T22:00:00", scale="utc"),
        Time("2026-04-26T11:00:00", scale="utc"),
    ),
    "2026-04-26": (
        Time("2026-04-26T22:00:00", scale="utc"),
        Time("2026-04-27T11:00:00", scale="utc"),
    ),
    "2026-04-27": (
        Time("2026-04-27T22:00:00", scale="utc"),
        Time("2026-04-28T11:00:00", scale="utc"),
    ),
}

t_start_12lou = Time("2026-04-23T22:00:00", scale="utc")
t_end_12lou = Time("2026-04-28T11:00:00", scale="utc")

# ── Fetch ─────────────────────────────────────────────────────────────────────
df_w12 = await client.select_time_series(
    flow_topic,
    fields=[outside_speed_field, outside_direction_field],
    start=t_start_12lou,
    end=t_end_12lou,
    index=weather_index,
)
df_d12 = await client.select_time_series(
    dome_topic,
    fields=[dome_azimuth_field],
    start=t_start_12lou,
    end=t_end_12lou,
)
df_s12 = await client.select_time_series(
    shutter_topic,
    fields=shutter_fields,
    start=t_start_12lou,
    end=t_end_12lou,
)
inside_12lou = {}
for idx, lbl in ess_sensors.items():
    inside_12lou[idx] = await client.select_time_series(
        turb_topic,
        fields=[speed_field],
        start=t_start_12lou,
        end=t_end_12lou,
        index=idx,
    )

print(f"Weather: {len(df_w12)} rows | Dome: {len(df_d12)} | Shutter: {len(df_s12)}")
for idx, lbl in ess_sensors.items():
    print(f"  Inside {idx:3d} {lbl:16s}: {len(inside_12lou[idx])} rows")

In [ ]:
# ── Align to 30-s grid ────────────────────────────────────────────────────────
w12_al = pd.DataFrame(
    {
        "outside_wind_speed_mps": df_w12[outside_speed_field]
        .astype(float)
        .resample(RESAMPLE_RULE)
        .mean(),
        "outside_wind_dir_deg": resample_direction(
            df_w12[outside_direction_field], RESAMPLE_RULE
        ),
    }
)
d12_al = pd.DataFrame(
    {
        "dome_azimuth_deg": resample_direction(
            df_d12[dome_azimuth_field], RESAMPLE_RULE
        ),
    }
)
s12_al = (
    df_s12[shutter_fields]
    .astype(float)
    .resample(RESAMPLE_RULE)
    .mean()
    .rename(
        columns={
            "positionActual0": "shutter_left_pct",
            "positionActual1": "shutter_right_pct",
        }
    )
)
s12_al["shutter_open_fraction"] = (
    0.5 * (s12_al["shutter_left_pct"] + s12_al["shutter_right_pct"]) / 100.0
)

i12_series = []
for idx, lbl in ess_sensors.items():
    i12_series.append(
        prepare_inside_sensor(
            inside_12lou[idx],
            sensor_idx=idx,
            label=lbl.replace(" ", "_"),
            clip_threshold=speed_clip[idx],
            rule=RESAMPLE_RULE,
        )
    )
i12_al = pd.concat(i12_series, axis=1)
i12_al["inside_speed_mean_mps"] = i12_al.mean(axis=1, skipna=True)
i12_al["inside_speed_median_mps"] = i12_al.median(axis=1, skipna=True)
i12_al["inside_sensor_count"] = i12_al.notna().sum(axis=1)

mdf12 = pd.concat([w12_al, d12_al, s12_al, i12_al], axis=1).sort_index()
mdf12 = mdf12[mdf12["inside_sensor_count"] >= MIN_INSIDE_SENSORS].copy()

mdf12 = compute_effective_areas(
    mdf12,
    louver_map=louver_map_12,
    shutter_full_open_area_m2=SHUTTER_FULL_OPEN_AREA_M2,
    shutter_azimuth_offset_deg=SHUTTER_AZIMUTH_OFFSET_DEG,
)

# ── Label each row with its night ─────────────────────────────────────────────
mdf12["night"] = pd.NA
for night_label, (ns, ne) in nights_12lou.items():
    ns_pd = pd.Timestamp(ns.iso)
    ne_pd = pd.Timestamp(ne.iso)
    if mdf12.index.tz is not None:
        ns_pd = ns_pd.tz_localize("UTC")
        ne_pd = ne_pd.tz_localize("UTC")
    mask = (mdf12.index >= ns_pd) & (mdf12.index < ne_pd)
    mdf12.loc[mask, "night"] = night_label

print(f"Aligned samples: {len(mdf12)}")
print(mdf12.groupby("night", dropna=False).size().rename("n_samples"))

In [ ]:
# ── Efficiency vs relative wind direction: 12-louver nights ───────────────────
eff12 = mdf12.copy()
eff12["relative_wind_deg"] = wrap180(
    eff12["outside_wind_dir_deg"]
    - (eff12["dome_azimuth_deg"] + SHUTTER_AZIMUTH_OFFSET_DEG)
)

# Use median of top-ring sensors only (123–126), excluding TMA Platform (110)
topring_cols = [
    c
    for c in eff12.columns
    if any(c.startswith(f"{idx}_") for idx in [123, 124, 125, 126])
]
eff12["inside_speed_topring_mps"] = eff12[topring_cols].median(axis=1, skipna=True)
eff12["airflow_efficiency"] = (
    eff12["inside_speed_topring_mps"] / eff12["outside_wind_speed_mps"]
)

eff12 = eff12[
    np.isfinite(eff12["relative_wind_deg"])
    & np.isfinite(eff12["airflow_efficiency"])
    & np.isfinite(eff12["outside_wind_speed_mps"])
    & np.isfinite(eff12["shutter_open_fraction"])
    & (eff12["shutter_open_fraction"] > 0.95)
    & (eff12["outside_wind_speed_mps"] > 0.5)
    & (eff12["airflow_efficiency"] >= 0)
    & (eff12["airflow_efficiency"] <= 3.0)
    & eff12["night"].notna()
].copy()

fig, ax1 = plt.subplots(figsize=(13, 6))

sc = ax1.scatter(
    eff12["relative_wind_deg"],
    eff12["airflow_efficiency"],
    c=eff12["outside_wind_speed_mps"],
    cmap="viridis",
    s=8,
    alpha=0.35,
    zorder=2,
)

# 15° binned median across all 4 nights
bins = np.arange(-180, 181, 15)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_idx = np.digitize(eff12["relative_wind_deg"], bins) - 1
med12 = [
    eff12.loc[bin_idx == i, "airflow_efficiency"].median() for i in range(len(centers))
]
ax1.plot(centers, med12, color="black", lw=2.2, label="15° bin median", zorder=4)

ax1.set_xlabel("Relative wind direction (° from shutter, 0° = wind into shutter)")
ax1.set_ylabel(
    "Airflow efficiency (inside / outside speed)\n[top-ring sensors 123–126 median]"
)
ax1.set_xlim(-180, 180)
ax1.set_xticks(np.arange(-180, 181, 30))
ax1.set_ylim(0, None)
ax1.grid(True, alpha=0.3)

cbar = plt.colorbar(sc, ax=ax1)
cbar.set_label("Outside wind speed (m/s)")

# ── Right axis: 12-louver windward aperture model ─────────────────────────────
areas_12lou = active_areas_for(louver_map, ids_12lou)
laz_arr = louver_map["azimuth_dome_deg"].to_numpy(dtype=float)
wt_12lou_no_lws = compute_windward_total(
    wind_angles_raw[sort_idx], laz_arr, areas_12lou, S_NO_LWS
)

ax2 = ax1.twinx()
ax2.plot(
    wind_deg,
    shutter_no_lws,
    color="tab:cyan",
    lw=1.4,
    ls="--",
    label=f"Shutter only, no LWS ({S_NO_LWS:.0f} m²)",
    zorder=3,
)
ax2.plot(
    wind_deg,
    wt_12lou_no_lws,
    color="firebrick",
    lw=2.2,
    label=(
        f"Shutter (no LWS) + 12-louver config  "
        f"[{areas_12lou.sum():.0f}+{S_NO_LWS:.0f} = {areas_12lou.sum()+S_NO_LWS:.0f} m²]"
    ),
    zorder=3,
)
ax2.set_ylabel("Windward open area (m²)", color="firebrick")
ax2.tick_params(axis="y", labelcolor="firebrick")
ax2.set_ylim(0, None)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

ax1.set_title(
    "Airflow efficiency vs relative wind direction\n"
    "12-louver config (A1–N1 excl. F/G + E2, H2)  |  nights 2026-04-23, -25, -26, -27"
    "  |  inside = median(ESS 123–126)"
)
plt.tight_layout()
plt.show()

print(f"Top-ring columns used: {topring_cols}")
print(f"Total filtered points: {len(eff12)}")
print(f"Median efficiency: {eff12['airflow_efficiency'].median():.3f}")
print(
    f"P10-P90 efficiency: {eff12['airflow_efficiency'].quantile(0.10):.3f} – {eff12['airflow_efficiency'].quantile(0.90):.3f}"
)

In [ ]:
# ── Comparison: 15° bin medians for 6-louver vs 12-louver configs ─────────────
bins = np.arange(-180, 181, 15)
centers = 0.5 * (bins[:-1] + bins[1:])


def bin_stats(df, col="airflow_efficiency", angle_col="relative_wind_deg"):
    idx = np.digitize(df[angle_col], bins) - 1
    med = np.array([df.loc[idx == i, col].median() for i in range(len(centers))])
    q25 = np.array([df.loc[idx == i, col].quantile(0.25) for i in range(len(centers))])
    q75 = np.array([df.loc[idx == i, col].quantile(0.75) for i in range(len(centers))])
    return med, q25, q75


med6, q25_6, q75_6 = bin_stats(eff_df)
med12, q25_12, q75_12 = bin_stats(eff12)

fig, ax = plt.subplots(figsize=(13, 5))

# 6-louver
ax.fill_between(centers, q25_6, q75_6, alpha=0.18, color="tab:blue")
ax.plot(
    centers,
    med6,
    color="tab:blue",
    lw=2.2,
    marker="o",
    ms=5,
    label=f"6-louver  (B1,E1,E2,H1,H2,M1) — n={len(eff_df):,}",
)

# 12-louver
ax.fill_between(centers, q25_12, q75_12, alpha=0.18, color="tab:red")
ax.plot(
    centers,
    med12,
    color="tab:red",
    lw=2.2,
    marker="s",
    ms=5,
    label=f"12-louver (A1–N1 excl. F/G) — n={len(eff12):,}",
)

ax.axvline(0, color="gray", ls=":", lw=1.2)
ax.set_xlabel("Relative wind direction (° from shutter, 0° = wind into shutter)")
ax.set_ylabel(
    "Airflow efficiency\n(inside / outside wind speed, top-ring median ESS 123–126)"
)
ax.set_xlim(-180, 180)
ax.set_xticks(np.arange(-180, 181, 30))
ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_title(
    "15° bin medians — 6-louver vs 12-louver commissioning configs\n"
    "Shaded band = IQR (Q25–Q75)  |  inside = median(ESS 123–126)"
)
plt.tight_layout()
plt.show()

print(f"6-louver  median efficiency: {np.nanmedian(med6):.3f}")
print(f"12-louver median efficiency: {np.nanmedian(med12):.3f}")

## Airflow Efficiency: 12-Louver Nights (2026-04-23, -25, -26, -27)

Four nights with the expanded commissioning configuration:
**A1, B1, C1, D1, E1, E2, H1, H2, I1, L1, M1, N1** (~110 m² effective louver area).

Points are coloured by night; the 15° binned median is overlaid.
The right axis shows the 12-louver windward-aperture model for reference.

In [ ]:
# ── 12-louver config: override open_fraction from CSV (currently 6-louver) ───
ids_12lou = ["A1", "B1", "C1", "D1", "E1", "E2", "H1", "H2", "I1", "L1", "M1", "N1"]
louver_map_12 = louver_map.copy()
louver_map_12["open_fraction"] = np.where(
    louver_map_12["louver_id"].isin(ids_12lou), 1.0, 0.0
)
louver_eff_12lou = (
    louver_map_12["effective_area_m2"] * louver_map_12["open_fraction"]
).sum()
print(f"12-louver effective area: {louver_eff_12lou:.1f} m²")
print(
    f"  open ids: {', '.join(louver_map_12[louver_map_12['open_fraction']>0]['louver_id'])}"
)

# ── Night boundaries (UTC; Chilean nights, UTC-4) ────────────────────────────
nights_12lou = {
    "2026-04-23": (
        Time("2026-04-23T22:00:00", scale="utc"),
        Time("2026-04-24T11:00:00", scale="utc"),
    ),
    "2026-04-25": (
        Time("2026-04-25T22:00:00", scale="utc"),
        Time("2026-04-26T11:00:00", scale="utc"),
    ),
    "2026-04-26": (
        Time("2026-04-26T22:00:00", scale="utc"),
        Time("2026-04-27T11:00:00", scale="utc"),
    ),
    "2026-04-27": (
        Time("2026-04-27T22:00:00", scale="utc"),
        Time("2026-04-28T11:00:00", scale="utc"),
    ),
}

t_start_12lou = Time("2026-04-23T22:00:00", scale="utc")
t_end_12lou = Time("2026-04-28T11:00:00", scale="utc")

# ── Fetch ─────────────────────────────────────────────────────────────────────
df_w12 = await client.select_time_series(
    flow_topic,
    fields=[outside_speed_field, outside_direction_field],
    start=t_start_12lou,
    end=t_end_12lou,
    index=weather_index,
)
df_d12 = await client.select_time_series(
    dome_topic,
    fields=[dome_azimuth_field],
    start=t_start_12lou,
    end=t_end_12lou,
)
df_s12 = await client.select_time_series(
    shutter_topic,
    fields=shutter_fields,
    start=t_start_12lou,
    end=t_end_12lou,
)
inside_12lou = {}
for idx, lbl in ess_sensors.items():
    inside_12lou[idx] = await client.select_time_series(
        turb_topic,
        fields=[speed_field],
        start=t_start_12lou,
        end=t_end_12lou,
        index=idx,
    )

print(f"Weather: {len(df_w12)} rows | Dome: {len(df_d12)} | Shutter: {len(df_s12)}")
for idx, lbl in ess_sensors.items():
    print(f"  Inside {idx:3d} {lbl:16s}: {len(inside_12lou[idx])} rows")

In [ ]:
# ── Align to common 30-s grid ─────────────────────────────────────────────────
w12_al = pd.DataFrame(
    {
        "outside_wind_speed_mps": df_w12[outside_speed_field]
        .astype(float)
        .resample(RESAMPLE_RULE)
        .mean(),
        "outside_wind_dir_deg": resample_direction(
            df_w12[outside_direction_field], RESAMPLE_RULE
        ),
    }
)
d12_al = pd.DataFrame(
    {
        "dome_azimuth_deg": resample_direction(
            df_d12[dome_azimuth_field], RESAMPLE_RULE
        ),
    }
)
s12_al = (
    df_s12[shutter_fields]
    .astype(float)
    .resample(RESAMPLE_RULE)
    .mean()
    .rename(
        columns={
            "positionActual0": "shutter_left_pct",
            "positionActual1": "shutter_right_pct",
        }
    )
)
s12_al["shutter_open_fraction"] = (
    0.5 * (s12_al["shutter_left_pct"] + s12_al["shutter_right_pct"]) / 100.0
)

i12_series = []
for idx, lbl in ess_sensors.items():
    i12_series.append(
        prepare_inside_sensor(
            inside_12lou[idx],
            sensor_idx=idx,
            label=lbl.replace(" ", "_"),
            clip_threshold=speed_clip[idx],
            rule=RESAMPLE_RULE,
        )
    )
i12_al = pd.concat(i12_series, axis=1)
i12_al["inside_speed_mean_mps"] = i12_al.mean(axis=1, skipna=True)
i12_al["inside_speed_median_mps"] = i12_al.median(axis=1, skipna=True)
i12_al["inside_sensor_count"] = i12_al.notna().sum(axis=1)

mdf12 = pd.concat([w12_al, d12_al, s12_al, i12_al], axis=1).sort_index()
mdf12 = mdf12[mdf12["inside_sensor_count"] >= MIN_INSIDE_SENSORS].copy()

mdf12 = compute_effective_areas(
    mdf12,
    louver_map=louver_map_12,
    shutter_full_open_area_m2=SHUTTER_FULL_OPEN_AREA_M2,
    shutter_azimuth_offset_deg=SHUTTER_AZIMUTH_OFFSET_DEG,
)

# ── Label each row with its night ─────────────────────────────────────────────
mdf12["night"] = pd.NA
for night_label, (ns, ne) in nights_12lou.items():
    ns_pd = pd.Timestamp(ns.iso)
    ne_pd = pd.Timestamp(ne.iso)
    if mdf12.index.tz is not None:
        ns_pd = ns_pd.tz_localize("UTC")
        ne_pd = ne_pd.tz_localize("UTC")
    mask = (mdf12.index >= ns_pd) & (mdf12.index < ne_pd)
    mdf12.loc[mask, "night"] = night_label

print(f"Aligned samples: {len(mdf12)}")
print(mdf12.groupby("night", dropna=False).size().rename("n_samples"))

In [ ]:
# ── Efficiency vs relative wind direction: 12-louver nights ───────────────────
eff12 = mdf12.copy()
eff12["relative_wind_deg"] = wrap180(
    eff12["outside_wind_dir_deg"]
    - (eff12["dome_azimuth_deg"] + SHUTTER_AZIMUTH_OFFSET_DEG)
)
eff12["airflow_efficiency"] = (
    eff12["inside_speed_mean_mps"] / eff12["outside_wind_speed_mps"]
)

eff12 = eff12[
    np.isfinite(eff12["relative_wind_deg"])
    & np.isfinite(eff12["airflow_efficiency"])
    & np.isfinite(eff12["outside_wind_speed_mps"])
    & np.isfinite(eff12["shutter_open_fraction"])
    & (eff12["shutter_open_fraction"] > 0.95)
    & (eff12["outside_wind_speed_mps"] > 0.5)
    & (eff12["airflow_efficiency"] >= 0)
    & (eff12["airflow_efficiency"] <= 3.0)
    & eff12["night"].notna()
].copy()

night_colors = {
    "2026-04-23": "tab:blue",
    "2026-04-25": "tab:orange",
    "2026-04-26": "tab:green",
    "2026-04-27": "tab:red",
}

fig, ax1 = plt.subplots(figsize=(13, 6))

for night_label, color in night_colors.items():
    sub = eff12[eff12["night"] == night_label]
    if sub.empty:
        continue
    ax1.scatter(
        sub["relative_wind_deg"],
        sub["airflow_efficiency"],
        c=color,
        s=8,
        alpha=0.30,
        label=f"Night {night_label}  (n={len(sub)})",
        zorder=2,
    )

# 15° binned median across all 4 nights
bins = np.arange(-180, 181, 15)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_idx = np.digitize(eff12["relative_wind_deg"], bins) - 1
med12 = [
    eff12.loc[bin_idx == i, "airflow_efficiency"].median() for i in range(len(centers))
]
ax1.plot(
    centers, med12, color="black", lw=2.2, label="15° bin median (all nights)", zorder=4
)

ax1.set_xlabel("Relative wind direction (° from shutter, 0° = wind into shutter)")
ax1.set_ylabel("Airflow efficiency (inside / outside speed)")
ax1.set_xlim(-180, 180)
ax1.set_xticks(np.arange(-180, 181, 30))
ax1.set_ylim(0, None)
ax1.grid(True, alpha=0.3)

# ── Right axis: 12-louver aperture model ─────────────────────────────────────
laz_arr = louver_map["azimuth_dome_deg"].to_numpy(dtype=float)
areas_12lou = active_areas_for(louver_map, ids_12lou)

wt_12lou_no_lws = compute_windward_total(
    wind_angles_raw[sort_idx], laz_arr, areas_12lou, S_NO_LWS
)

ax2 = ax1.twinx()
ax2.plot(
    wind_deg,
    shutter_no_lws,
    color="tab:cyan",
    lw=1.4,
    ls="--",
    label=f"Shutter only, no LWS ({S_NO_LWS:.0f} m²)",
    zorder=3,
)
ax2.plot(
    wind_deg,
    wt_12lou_no_lws,
    color="firebrick",
    lw=2.2,
    label=f"Shutter (no LWS) + 12-louver config  [{areas_12lou.sum():.0f}+{S_NO_LWS:.0f} = {areas_12lou.sum()+S_NO_LWS:.0f} m²]",
    zorder=3,
)
ax2.set_ylabel("Windward open area (m²)", color="firebrick")
ax2.tick_params(axis="y", labelcolor="firebrick")
ax2.set_ylim(0, None)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

ax1.set_title(
    "Airflow efficiency vs relative wind direction\n"
    f"12-louver config (A1–N1 excl. F/G + E2, H2) | nights 2026-04-23, -25, -26, -27"
)
plt.tight_layout()
plt.show()

print(f"Total filtered points: {len(eff12)}")
stats = (
    eff12.groupby("night")["airflow_efficiency"]
    .describe()[["count", "mean", "50%", "std"]]
    .rename(columns={"50%": "median"})
    .round(3)
)
print(stats)